# 04 – Corpus audit (deduplication, on-topic filter, data copies)

Checks on the frozen news corpus (`data/news_clean.csv`, 420 articles; see `dataset/news/corpus_manifest.json`). **Nothing here changes the corpus.** Findings are reported; any change would be a new, documented corpus version.

1. What the deduplication step removed (504 → 420): rule, before/after tables, and whether the removed articles are syndicated wire copy.
2. Whether near-duplicates remain in the 420, at stricter similarity thresholds than the rule used.
3. Whether the on-topic filter treats upper/lower case the same way as the frame scorer.
4. The three copies of the 90 MB dataset file: identical or not, and which one the code reads.

In [1]:
import hashlib, json, re
from pathlib import Path

import numpy as np
import pandas as pd
import tldextract
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

OUT = Path("dataset/analysis"); OUT.mkdir(parents=True, exist_ok=True)
CFG = json.load(open("config.json"))
SCRAPED, URLS, CORPUS = Path(CFG["scraped_csv"]), Path(CFG["urls_full_csv"]), Path(CFG["news_corpus_csv"])
MIN_WORDS, MAX_WORDS = 150, 3000
registered = tldextract.TLDExtract(suffix_list_urls=())      # offline: no network lookup of the suffix list

## 1. Rebuild the cleaning steps exactly as `news_corpus_v2.ipynb` ran them
Same order: usable text, ≤ 3,000 words, on-topic (≥ 2 AI words and ≥ 2 regulation words), then two duplicate rules: (a) identical text after lower-casing and stripping punctuation, (b) identical non-empty title. The rebuilt numbers must match the corpus log, otherwise this audit would be about a different corpus.

In [2]:
AI_WORDS  = re.compile(r"\b(AI|artificial intelligence|chatbots?|ChatGPT|large language models?|LLMs?|OpenAI|Gemini)\b")
REG_WORDS = re.compile(r"\b(regulat\w*|legislat\w*|laws?|AI Act|bills?|rules?|oversight|ban|governance|policy|lawmakers?)\b", re.I)

scraped = pd.read_csv(SCRAPED, keep_default_na=False)
urls = pd.read_csv(URLS)
news = scraped.merge(urls[["url", "outlet", "region", "month", "quarter"]], on="url")
news = news[(news["status"] == "ok") & (news["n_words"].astype(float) <= MAX_WORDS)].copy()
news["ai_hits"]  = news["text"].map(lambda t: len(AI_WORDS.findall(t)))
news["reg_hits"] = news["text"].map(lambda t: len(REG_WORDS.findall(t)))
on_topic = news[(news["ai_hits"] >= 2) & (news["reg_hits"] >= 2)].copy().reset_index(drop=True)
on_topic["text_key"] = on_topic["text"].str.lower().str.replace(r"\W+", " ", regex=True).str.strip()

# rule (a): same normalised text -> keep the first; rule (b): same non-empty title -> keep the first
on_topic["dup_rule"] = ""
first_text = on_topic.drop_duplicates("text_key").index
on_topic.loc[~on_topic.index.isin(first_text), "dup_rule"] = "a: identical text"
kept_a = on_topic[on_topic["dup_rule"] == ""]
is_title_dup = (kept_a["title"] != "") & kept_a.duplicated("title")
on_topic.loc[is_title_dup[is_title_dup].index, "dup_rule"] = "b: identical title"

after = on_topic[on_topic["dup_rule"] == ""]
log = json.load(open("dataset/news/corpus_log.json"))
logged = dict(log["cleaning_steps"])
print("rebuilt  :", len(on_topic), "on-topic ->", len(after), "after de-duplication")
print("logged   :", logged["on-topic"], "->", logged["no duplicates"])
assert (len(on_topic), len(after)) == (logged["on-topic"], logged["no duplicates"]), "rebuild does not match the corpus log"
assert set(after["url"]) == set(pd.read_csv(CORPUS, keep_default_na=False)["url"]), "rebuilt set differs from the frozen corpus"
print("rebuilt set is identical to the frozen corpus (by URL)")
print(on_topic["dup_rule"].value_counts().to_dict())

rebuilt  : 504 on-topic -> 420 after de-duplication
logged   : 504 -> 420
rebuilt set is identical to the frozen corpus (by URL)
{'': 420, 'b: identical title': 43, 'a: identical text': 41}


## 2. Before / after tables

In [3]:
before_after = pd.concat({"before": on_topic.groupby("region").size(), "after": after.groupby("region").size()}, axis=1).fillna(0).astype(int)
before_after["removed"] = before_after["before"] - before_after["after"]
before_after["removed_%"] = (100 * before_after["removed"] / before_after["before"]).round(1)
before_after.loc["All"] = before_after.sum(numeric_only=True); before_after.loc["All", "removed_%"] = round(100 * before_after.loc["All", "removed"] / before_after.loc["All", "before"], 1)
before_after.to_csv(OUT / "dedup_before_after_by_region.csv")
display(before_after)

outlets = pd.concat({"before": on_topic.groupby("outlet").size(), "after": after.groupby("outlet").size()}, axis=1).fillna(0).astype(int)
outlets["removed"] = outlets["before"] - outlets["after"]
outlets.sort_values("removed", ascending=False).to_csv(OUT / "dedup_before_after_by_outlet.csv")
print("outlets before:", (outlets["before"] > 0).sum(), "| after:", (outlets["after"] > 0).sum(), "| lost entirely:", ((outlets["before"] > 0) & (outlets["after"] == 0)).sum())
print("largest outlet before:", outlets["before"].max(), "| after:", outlets["after"].max())
outlets.sort_values("removed", ascending=False).head(10)

,before,after,removed,removed_%
region,,,,
EU,94.0,82.0,12.0,12.8
Other,117.0,104.0,13.0,11.1
UK,124.0,101.0,23.0,18.5
US,169.0,133.0,36.0,21.3
All,504.0,420.0,84.0,16.7


outlets before: 346 | after: 285 | lost entirely: 61
largest outlet before: 11 | after: 11


,before,after,removed
outlet,,,
winnipegfreepress.com,3,0,3
californiatelegraph.com,3,1,2
wysu.org,2,0,2
digitaljournal.com,4,2,2
theusnews.com,2,0,2
chinanationalnews.com,1,0,1
kacu.org,1,0,1
news4jax.com,1,0,1
thehindubusinessline.com,3,2,1


In [4]:
# Outlet labels: ports (:443) and other variants of the same site would be counted as different outlets
corpus_o = pd.read_csv(CORPUS, keep_default_na=False)
odd = corpus_o[corpus_o["outlet"].str.contains(r":\d+$|^www\.", regex=True)]
cleaned = corpus_o["outlet"].str.replace(r":\d+$", "", regex=True).str.replace(r"^www\.", "", regex=True)
print("outlet labels with a port or www:", len(odd), "->", sorted(set(odd["outlet"])))
print("distinct outlets as labelled:", corpus_o["outlet"].nunique(), "| after stripping port/www:", cleaned.nunique(),
      "| distinct registered domains:", corpus_o["url"].map(lambda u: registered(u).top_domain_under_public_suffix).nunique())

outlet labels with a port or www: 3 -> ['ksl.com:443', 'mondaq.com:443']
distinct outlets as labelled: 285 | after stripping port/www: 285 | distinct registered domains: 283


## 3. Are the removed articles syndicated wire copy?
For every removed article, find the kept article it duplicates and compare: same outlet, same site family (registered domain, e.g. several `iheart.com` stations), different outlet, and whether a wire-agency name appears at the start or end of the text.

In [5]:
WIRE = re.compile(r"\b(Reuters|Associated Press|AP|AFP|Bloomberg|PTI|IANS|ANI|Xinhua|PA Media|Press Association|UPI|Agence France-Presse)\b")
fam = lambda u: registered(u).top_domain_under_public_suffix or u

kept = after.set_index("text_key")["url"].to_dict()
kept_title = after[after["title"] != ""].set_index("title")["url"].to_dict()
by_url = on_topic.set_index("url")

rows = []
for r in on_topic[on_topic["dup_rule"] != ""].itertuples():
    twin_url = kept.get(r.text_key) if r.dup_rule.startswith("a") else kept_title.get(r.title)
    t = by_url.loc[twin_url] if twin_url in by_url.index else None
    sim = np.nan
    if t is not None:
        v = TfidfVectorizer().fit_transform([r.text, t["text"]]); sim = float(cosine_similarity(v[0], v[1])[0, 0])
    edge = r.text[:300] + " " + r.text[-300:]
    rows.append({"removed_url": r.url, "removed_outlet": r.outlet, "region": r.region, "rule": r.dup_rule, "kept_url": twin_url,
                 "kept_outlet": t["outlet"] if t is not None else None, "text_cosine": round(sim, 3) if sim == sim else None,
                 "same_outlet": t is not None and t["outlet"] == r.outlet,
                 "same_site_family": t is not None and fam(twin_url) == fam(r.url),
                 "wire_agency_named": bool(WIRE.search(edge)), "title": r.title[:90]})
removed = pd.DataFrame(rows)
removed["kind"] = np.select([removed["same_outlet"], removed["same_site_family"], removed["wire_agency_named"]],
                            ["same outlet (re-posted)", "same site family (network of stations/sites)", "different outlet, wire agency named"],
                            "different outlet, no agency named")
removed.to_csv(OUT / "dedup_removed_articles.csv", index=False)
print(len(removed), "removed")
display(removed["kind"].value_counts().to_frame("articles"))
display(removed.groupby("rule")["text_cosine"].describe().round(3))
display(removed.groupby("region")["kind"].value_counts().unstack(fill_value=0))
removed.sample(8, random_state=1)[["removed_outlet", "kept_outlet", "rule", "text_cosine", "kind", "title"]]

84 removed


,articles
kind,
"different outlet, no agency named",58
"different outlet, wire agency named",25
same site family (network of stations/sites),1


,count,mean,std,min,25%,50%,75%,max
rule,,,,,,,,
a: identical text,35.0,1.000,0.000,1.000,1.000,1.000,1.000,1.0
b: identical title,43.0,0.946,0.046,0.839,0.915,0.947,0.992,1.0


kind,"different outlet, no agency named","different outlet, wire agency named",same site family (network of stations/sites)
region,,,
EU,8,4,0
Other,5,8,0
UK,12,10,1
US,33,3,0


,removed_outlet,kept_outlet,rule,text_cosine,kind,title
10,tennesseedaily.com,indiagazette.com,b: identical title,0.947,"different outlet, no agency named","Pharma firms use AI to streamline trials, regu..."
67,chinanationalnews.com,indiagazette.com,b: identical title,0.903,"different outlet, no agency named","Pharma firms use AI to streamline trials, regu..."
59,pittsburghstar.com,baltimorestar.com,b: identical title,0.962,"different outlet, no agency named",Resemble AI Raises $13M From Sony Innovation F...
33,pasadenastarnews.com,abc13.com,b: identical title,0.947,"different outlet, wire agency named",Trump signs an executive order to vet top AI m...
83,lawgazette.co.uk:443,lawgazette.co.uk,a: identical text,1.000,same site family (network of stations/sites),Agentic AI: team player
77,news4jax.com,ksat.com,a: identical text,1.000,"different outlet, wire agency named",Musk's AI chatbot faces global backlash over s...
34,capeandislands.org,wnmufm.org,a: identical text,1.000,"different outlet, no agency named",Trump's new AI policies keep culture war focus...
38,dallassun.com,manilatimes.net,b: identical title,0.986,"different outlet, no agency named",The AI Support Paradox: Why Security-Conscious...


## 4. Do near-duplicates remain in the 420?
The deduplication rule only caught *identical* text or title. Near-copies (a wire story with a different headline, a lightly edited rewrite) would pass. Two similarity measures on the 420, both stricter than the rule:
- TF-IDF cosine on the **full text** (≥ 0.80 strict tier, 0.60–0.80 borderline tier),
- word 5-gram Jaccard on the **first 300 words**, which is the text the frame scorer actually uses (≥ 0.50 strict tier).

In [6]:
corpus = pd.read_csv(CORPUS, keep_default_na=False)
X = TfidfVectorizer(sublinear_tf=True, stop_words="english").fit_transform(corpus["text"])
cos = cosine_similarity(X); np.fill_diagonal(cos, 0)

def shingles(t, k=5):
    w = re.sub(r"\W+", " ", t.lower()).split()[:300]
    return {" ".join(w[i:i + k]) for i in range(max(len(w) - k + 1, 1))}
S = [shingles(t) for t in corpus["text"]]

pairs = []
for i, j in zip(*np.where(np.triu(cos) >= 0.60)):
    jac = len(S[i] & S[j]) / max(len(S[i] | S[j]), 1)
    pairs.append((i, j, cos[i, j], jac))
for i in range(len(S)):                                              # Jaccard catches pairs that cosine misses
    for j in range(i + 1, len(S)):
        if (i, j) in {(p[0], p[1]) for p in pairs}: continue
        jac = len(S[i] & S[j]) / max(len(S[i] | S[j]), 1)
        if jac >= 0.50: pairs.append((i, j, cos[i, j], jac))

near = pd.DataFrame([{"doc_a": corpus.loc[i, "doc_id"], "doc_b": corpus.loc[j, "doc_id"], "outlet_a": corpus.loc[i, "outlet"], "outlet_b": corpus.loc[j, "outlet"],
                      "region_a": corpus.loc[i, "region"], "region_b": corpus.loc[j, "region"], "cosine_full_text": round(c, 3), "jaccard_5gram_first300": round(jc, 3),
                      "tier": "strict" if (c >= 0.80 or jc >= 0.50) else "borderline", "title_a": corpus.loc[i, "title"][:70], "title_b": corpus.loc[j, "title"][:70]}
                     for i, j, c, jc in pairs]).sort_values("cosine_full_text", ascending=False)
near.to_csv(OUT / "near_duplicates_remaining.csv", index=False)
print("strict-tier pairs:", (near["tier"] == "strict").sum() if len(near) else 0, "| borderline pairs:", (near["tier"] == "borderline").sum() if len(near) else 0)
near if len(near) else print("none")

strict-tier pairs: 25 | borderline pairs: 12


,doc_a,doc_b,outlet_a,outlet_b,region_a,region_b,cosine_full_text,jaccard_5gram_first300,tier,title_a,title_b
33,news_0348,news_0362,mercurynews.com,bostonglobe.com,EU,EU,0.993,0.941,strict,Musk’s AI chatbot faces global backlash over s...,Musk’s AI chatbot faces global backlash over s...
29,news_0346,news_0348,katu.com,mercurynews.com,EU,EU,0.992,0.922,strict,Global backlash mounts against Musk's AI over ...,Musk’s AI chatbot faces global backlash over s...
31,news_0346,news_0362,katu.com,bostonglobe.com,EU,EU,0.991,0.885,strict,Global backlash mounts against Musk's AI over ...,Musk’s AI chatbot faces global backlash over s...
23,news_0315,news_0349,lockhaven.com,whdh.com,EU,UK,0.991,0.993,strict,Musk’s AI chatbot faces global backlash over s...,Musk’s AI chatbot faces global backlash over s...
5,news_0037,news_0244,readingeagle.com,fortune.com,US,US,0.986,0.928,strict,Regulators struggle to keep up with the fast-m...,People are talking to this AI startup's cartoo...
35,news_0357,news_0363,cronkitenews.azpbs.org,azbigmedia.com,EU,EU,0.985,0.816,strict,"As more lawyers fall for AI hallucinations, Ch...",How AI-driven hallucinatory filings are impact...
24,news_0315,news_0362,lockhaven.com,bostonglobe.com,EU,EU,0.984,0.954,strict,Musk’s AI chatbot faces global backlash over s...,Musk’s AI chatbot faces global backlash over s...
22,news_0315,news_0348,lockhaven.com,mercurynews.com,EU,EU,0.980,0.980,strict,Musk’s AI chatbot faces global backlash over s...,Musk’s AI chatbot faces global backlash over s...
21,news_0315,news_0346,lockhaven.com,katu.com,EU,EU,0.978,0.922,strict,Musk’s AI chatbot faces global backlash over s...,Global backlash mounts against Musk's AI over ...
34,news_0349,news_0362,whdh.com,bostonglobe.com,UK,EU,0.978,0.960,strict,Musk’s AI chatbot faces global backlash over s...,Musk’s AI chatbot faces global backlash over s...


Pairs link into **clusters** (one wire story in several outlets). Connected components on the strict tier show how many *distinct* stories the 420 really contain.

In [7]:
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
strict = near[near["tier"] == "strict"]
ids = sorted(set(strict["doc_a"]) | set(strict["doc_b"])); ix = {d: k for k, d in enumerate(ids)}
rows_, cols_ = [ix[d] for d in strict["doc_a"]], [ix[d] for d in strict["doc_b"]]
_, lab = connected_components(coo_matrix((np.ones(len(rows_)), (rows_, cols_)), shape=(len(ids), len(ids))), directed=False)
comps = [sorted(d for d, l in zip(ids, lab) if l == k) for k in sorted(set(lab))]
info = corpus.set_index("doc_id")
cl = pd.DataFrame([{"cluster": k + 1, "size": len(c), "docs": ", ".join(c), "outlets": ", ".join(info.loc[c, "outlet"]),
                    "regions": ", ".join(sorted(set(info.loc[c, "region"]))), "example_title": info.loc[c[0], "title"][:80]} for k, c in enumerate(sorted(comps, key=len, reverse=True))])
cl.to_csv(OUT / "near_duplicate_clusters.csv", index=False)
n_in = sum(len(c) for c in comps); extra = n_in - len(comps)
print(len(comps), "clusters contain", n_in, "articles; keeping one per cluster would drop", extra, "of 420 (", round(100 * extra / 420, 1), "%)")
drop = [d for c in comps for d in c[1:]]
print("articles that would drop, by region:", info.loc[drop, "region"].value_counts().to_dict(), "| frozen corpus by region:", info["region"].value_counts().to_dict())
cl

14 clusters contain 33 articles; keeping one per cluster would drop 19 of 420 ( 4.5 %)
articles that would drop, by region: {'EU': 7, 'US': 6, 'Other': 3, 'UK': 3} | frozen corpus by region: {'US': 133, 'Other': 104, 'UK': 101, 'EU': 82}


,cluster,size,docs,outlets,regions,example_title
0,1,5,"news_0315, news_0346, news_0348, news_0349, ne...","lockhaven.com, katu.com, mercurynews.com, whdh...","EU, UK",Musk’s AI chatbot faces global backlash over s...
1,2,3,"news_0002, news_0085, news_0268","turnto23.com, abc13.com, independent.co.uk",US,Trump directs all government agencies to stop ...
2,3,3,"news_0008, news_0039, news_0122","wclk.com, mb.com.ph, mysuncoast.com","EU, Other","Malaysia, Indonesia become first to block Musk..."
3,4,2,"news_0037, news_0244","readingeagle.com, fortune.com",US,Regulators struggle to keep up with the fast-m...
4,5,2,"news_0062, news_0209","indiagazette.com, bruneinews.net",EU,"Pharma firms use AI to streamline trials, regu..."
5,6,2,"news_0069, news_0263","clickondetroit.com, denverpost.com",UK,Musk's Grok chatbot restricts image generation...
6,7,2,"news_0125, news_0126","lockhaven.com, thetimes-tribune.com",US,"As Pennsylvania cracks down on AI, multiple ch..."
7,8,2,"news_0195, news_0205","dawn.com, cbsnews.com",Other,TECHNOLOGY: THE UNCOMFORTABLE TRUTH ABOUT AI
8,9,2,"news_0196, news_0248","nbcbayarea.com, durangoherald.com",US,Trump's order to block ‘woke' AI in government...
9,10,2,"news_0202, news_0269","inquirer.com, newsday.com",US,Anthropic CEO says AI company 'cannot in good ...


## 5. Does the on-topic filter use the same casing as the frame scorer?
The frame scorer lower-cases text before matching (`03_frame_scoring.ipynb`). The on-topic filter used in `news_corpus_v2.ipynb` is a separate regex scorer: its regulation list is case-insensitive, but its AI list is **case-sensitive** (`AI`, `ChatGPT`, `OpenAI`, `Gemini`, and lower-case multi-word terms such as `artificial intelligence`). So a sentence that *starts* with "Artificial intelligence" or "Chatbots" would not count. The corpus is frozen, so this is only measured: how many articles would change status under a case-insensitive AI list (keeping `AI` itself case-sensitive so it does not match the word "ai" inside other languages or names)?

In [8]:
AI_CI = re.compile(r"\b(AI|artificial intelligence|chatbots?|ChatGPT|large language models?|LLMs?|OpenAI|Gemini)\b", re.I)
AI_MIXED = re.compile(r"\bAI\b|\b(?i:artificial intelligence|chatbots?|chatgpt|large language models?|llms?|openai|gemini)\b")
sc = news.copy()
sc["ai_mixed"] = sc["text"].map(lambda t: len(AI_MIXED.findall(t)))
now  = (sc["ai_hits"] >= 2) & (sc["reg_hits"] >= 2)
alt  = (sc["ai_mixed"] >= 2) & (sc["reg_hits"] >= 2)
print("on-topic now:", int(now.sum()), "| with case-insensitive multi-word AI terms:", int(alt.sum()), "| would be added:", int((alt & ~now).sum()), "| would be lost:", int((now & ~alt).sum()))
print("share of the 504 on-topic articles with at least one AI-term hit that only the case-insensitive list sees:", round(((sc["ai_mixed"] > sc["ai_hits"]) & now).sum() / now.sum(), 3))

on-topic now: 504 | with case-insensitive multi-word AI terms: 504 | would be added: 0 | would be lost: 0
share of the 504 on-topic articles with at least one AI-term hit that only the case-insensitive list sees: 0.26


## 6. The three copies of the 90 MB dataset file

In [9]:
def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

home = Path.home()
copies = {"repo (code reads this one)": Path(CFG["parquet"]),
          "Desktop/NLP_CA/data": home / "Desktop/NLP_CA/data/genai_gdelt_articles.parquet",
          "Downloads/data": home / "Downloads/data/genai_gdelt_articles.parquet"}
rows = [{"copy": k, "exists": p.exists(), "bytes": p.stat().st_size if p.exists() else None, "sha256": sha(p)[:16] if p.exists() else None} for k, p in copies.items()]
cp = pd.DataFrame(rows); cp.to_csv(OUT / "dataset_copies_hashes.csv", index=False)
display(cp)
print("all identical:", cp["sha256"].nunique() == 1)

old = {"Desktop/NLP_CA/data": home / "Desktop/NLP_CA/data", "Downloads/data": home / "Downloads/data"}
for k, d in old.items():
    print(k, "->", sorted(p.name for p in d.iterdir() if p.is_file() and p.name != "genai_gdelt_articles.parquet"))

,copy,exists,bytes,sha256
0,repo (code reads this one),True,90425481,04759d58c2f7392a
1,Desktop/NLP_CA/data,True,90425481,04759d58c2f7392a
2,Downloads/data,True,90425481,04759d58c2f7392a


all identical: True
Desktop/NLP_CA/data -> ['corpus_log.json', 'news_clean.csv', 'scraped.csv', 'urls_full.csv', 'urls_pilot.csv']
Downloads/data -> ['corpus_log.json', 'news_clean.csv', 'scraped.csv', 'urls_full.csv', 'urls_pilot.csv']


In [10]:
# Do any notebooks or scripts read the old locations or the old 45-article files?
import glob
hits = []
for f in sorted(glob.glob("*.ipynb") + glob.glob("*.py")):
    src = "".join("".join(c["source"]) for c in json.load(open(f))["cells"] if c["cell_type"] == "code") if f.endswith(".ipynb") else open(f).read()
    for pat in ["Desktop/NLP_CA", "Downloads", "/Users/"]:
        if pat in src: hits.append((f, "absolute path: " + pat))
    for m in re.finditer(r"[\w./-]*(?:news_clean|scraped|urls_full|urls_pilot|corpus_log)[\w./-]*", src):
        hits.append((f, m.group(0)))
pd.DataFrame(hits, columns=["file", "reference"]).drop_duplicates().sort_values("file")

,file,reference
0,04_corpus_audit.ipynb,absolute path: Desktop/NLP_CA
12,04_corpus_audit.ipynb,corpus_log
11,04_corpus_audit.ipynb,urls_pilot
8,04_corpus_audit.ipynb,news_clean
7,04_corpus_audit.ipynb,dataset/news/corpus_log.json
6,04_corpus_audit.ipynb,scraped.merge
10,04_corpus_audit.ipynb,urls_full
4,04_corpus_audit.ipynb,urls_full_csv
3,04_corpus_audit.ipynb,scraped_csv
2,04_corpus_audit.ipynb,absolute path: /Users/
